# 🌾 Agronomist Synthetic Data Generator

A lightweight tool for generating **realistic, fully synthetic agricultural datasets** in **CSV or JSON format**. Designed for agronomy workflows, testing pipelines, and prototyping data-driven applications—without exposing real-world farm data.

---

## 🚀 Overview

This notebook provides a structured system for generating agriculture-focused datasets using predefined schemas or custom user-defined scenarios. Outputs are clean, machine-ready, and strictly formatted.

---

## ⚙️ Key Features

* **Strict Output Control**

  * CSV or JSON only
  * No explanations, comments, or Markdown in generated data

* **Safe Synthetic Data**

  * Realistic but entirely fictional
  * No real farms, regions, or identifiable locations

* **Template-Based Generation**

  * Multiple agronomy-specific schemas
  * Optional fully custom dataset generation

* **Interactive UI**

  * Built with Gradio for easy dataset generation

---

## 🧠 System Prompt Design

The system prompt enforces:

* **Data-only responses**
* **Format validation** (CSV / JSON only)
* **Refusal handling** for:

  * Unsupported formats
  * Non-dataset requests
* **Agriculture constraints**:

  * Synthetic realism
  * No real-world identifiers

---

## 📊 Supported Dataset Templates

### 1. Crop Records

Tracks crop production and field-level yield data.

**Fields:**

* `crop_name`
* `variety`
* `planting_date`
* `field_id`
* `yield_kg_per_ha`
* `region`
* `season`

---

### 2. Soil Samples

Captures soil health and composition metrics.

**Fields:**

* `sample_id`
* `field_id`
* `ph`
* `organic_matter_pct`
* `n_p_k`
* `texture`
* `sample_date`

---

### 3. Weather / Field Conditions

Records environmental conditions affecting crops.

**Fields:**

* `date`
* `field_id`
* `temp_c`
* `rainfall_mm`
* `humidity_pct`
* `location`

---

### 4. Pest & Disease Records

Logs crop health threats and treatments.

**Fields:**

* `record_id`
* `field_id`
* `crop`
* `pest_or_disease`
* `severity`
* `observed_date`
* `treatment`

---

### 5. Harvest Logs

Tracks yield quantity and quality at harvest.

**Fields:**

* `harvest_id`
* `crop`
* `field_id`
* `quantity_kg`
* `grade`
* `harvest_date`

---

### 6. Irrigation Records

Monitors irrigation practices and water usage.

**Fields:**

* `field_id`
* `date`
* `method`
* `volume_m3`
* `duration_min`

---

### 7. Custom Dataset

* Schema is inferred dynamically
* Driven by user-provided agricultural instructions

---

## 🏗️ Core Functions

### `build_user_prompt()`

Constructs a structured prompt using:

* Selected dataset template
* Desired number of rows
* Output format (CSV or JSON)

---

### `generate_dataset()`

Handles dataset generation by:

1. Calling the OpenAI model (`gpt-4o-mini`)
2. Applying system and user prompts
3. Cleaning output:

   * Removes Markdown code fences if present
4. Returning:

   * Raw CSV or JSON
   * Or a refusal/error if applicable

---

## 🖥️ Gradio Interface

### Inputs

* **Dataset Type**

  * Dropdown of predefined templates

* **Number of Rows**

  * Range: 1–500 (default: 10)

* **Output Format**

  * CSV or JSON

* **Custom Instruction**

  * Enabled only when *Custom* dataset type is selected

---

### Output

* Generated dataset displayed in a textbox
* Ready for:

  * Copy/paste
  * Manual saving

---

## 🧪 Use Cases

* Testing data pipelines
* Prototyping agritech applications
* Training/demo datasets
* Educational simulations

---

## 📌 Notes

* This tool is **not intended for real agricultural reporting**
* All generated data is synthetic and should be treated as such
* Output cleanliness is prioritized for direct downstream use

---

If you want, I can also:

* Add installation/setup instructions
* Turn this into a full GitHub repo structure
* Or generate example outputs for each template


# Agronomist Synthetic Data Generator

Generate structured CSV or JSON datasets for agriculture use cases: crop records, soil samples, weather/field conditions, pest & disease logs, harvest data, and irrigation. Uses OpenAI and Gradio; output is data-only (no explanations).

In [ ]:
# Imports
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Set OPENAI_API_KEY in your .env file.")
client = OpenAI(api_key=api_key)



In [ ]:
# System prompt: strict structured output only (CSV or JSON)
SYSTEM_PROMPT = """
ROLE:
You are a synthetic dataset generator.

PRIMARY FUNCTION:
Generate ONLY structured semantic data.

ALLOWED OUTPUT FORMATS:
- CSV (default)
- JSON (only if explicitly requested)

STRICT CONSTRAINTS:
- Output must be structured data ONLY.
- Do NOT provide explanations, reasoning, commentary, or natural language responses.
- Do NOT answer questions, provide analysis, or generate instructional content.
- Do NOT use Markdown or any formatting wrappers.
- Output must be directly usable as raw CSV or JSON.

FORMAT RULES:
- If the user does not specify a format, output CSV.
- If the user explicitly requests JSON, output JSON.
- If the user requests any other format (e.g., YAML, XML, Markdown, text, tables, prose), you MUST refuse.

REFUSAL BEHAVIOR:
- If the request violates format or scope rules, respond with a single-line refusal message.
- Do NOT include explanations or additional text.

STANDARD REFUSAL MESSAGES:
- For unsupported formats:
  "Unsupported format. Allowed formats: CSV or JSON."
- For unstructured output or question answering:
  "Unsupported request. Only structured dataset generation is allowed."

Generate realistic but fake agriculture data. No real locations, farm names, or identifiable information.
Generate exactly the number of records requested.
"""

In [ ]:
# Agronomist prompt templates: (schema description, generation instructions)
PROMPT_TEMPLATES = {
    "Crop records": (
        "Each record must have: crop_name, variety, planting_date (YYYY-MM-DD), field_id, yield_kg_per_ha (number), region, season.",
        "Generate realistic but fake crop records. Mix crops and regions; vary yields and planting dates."
    ),
    "Soil samples": (
        "Each record must have: sample_id, field_id, ph (number), organic_matter_pct (number), n_p_k (e.g. 10-20-10 or three numbers), texture (e.g. loam/sandy/clay), sample_date (YYYY-MM-DD).",
        "Generate soil sample entries. Use plausible pH and nutrient ranges; vary textures and dates."
    ),
    "Weather / field conditions": (
        "Each record must have: date (YYYY-MM-DD), field_id, temp_c (number), rainfall_mm (number), humidity_pct (number), location (fake place name).",
        "Generate daily field condition rows. Vary temperature, rainfall, and humidity; keep values realistic."
    ),
    "Pest & disease records": (
        "Each record must have: record_id, field_id, crop, pest_or_disease (name), severity (low/medium/high), observed_date (YYYY-MM-DD), treatment (short description or none).",
        "Generate pest and disease log entries. Mix severities and treatments; use plausible fake names."
    ),
    "Harvest logs": (
        "Each record must have: harvest_id, crop, field_id, quantity_kg (number), grade (e.g. A/B/C or 1/2/3), harvest_date (YYYY-MM-DD).",
        "Generate harvest log rows. Vary quantities and grades; use consistent crop and field IDs."
    ),
    "Irrigation": (
        "Each record must have: field_id, date (YYYY-MM-DD), method (drip/sprinkler/flood/etc), volume_m3 (number), duration_min (number).",
        "Generate irrigation event records. Vary methods and volumes; keep dates and durations realistic."
    ),
    "Custom": (
        "Infer a sensible schema from the user's scenario. Each record must be a JSON object with consistent keys across all records.",
        "Generate synthetic agriculture-related data that matches the user's description. No real PII; use plausible fake values."
    ),
}

In [ ]:
def build_user_prompt(template_name: str, num_rows: int, output_format: str, custom_instruction: str = "") -> str:
    """Build the user prompt from template (or custom) and parameters."""
    if template_name not in PROMPT_TEMPLATES:
        template_name = "Custom"
    schema, instructions = PROMPT_TEMPLATES[template_name]
    fmt = "JSON" if output_format.lower() == "json" else "CSV"
    if template_name == "Custom" and custom_instruction.strip():
        return f"Generate {num_rows} rows of synthetic data. Scenario: {custom_instruction.strip()}. Output format: {fmt}. Schema: {schema}. Instructions: {instructions}"
    return f"Generate {num_rows} rows. Schema: {schema}. Instructions: {instructions}. Output format: {fmt}."

In [ ]:
def generate_dataset(template_name: str, num_rows: int, output_format: str, custom_instruction: str = "") -> str:
    """Call OpenAI and return raw data or refusal message."""
    user_prompt = build_user_prompt(template_name, num_rows, output_format, custom_instruction)
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.3,
        )
        raw = (response.choices[0].message.content or "").strip()
        # Remove common markdown wrappers if present (model sometimes ignores system prompt)
        for wrapper in ("```csv", "```json", "```"):
            if raw.startswith(wrapper):
                raw = raw[len(wrapper):].lstrip()
            if raw.endswith("```"):
                raw = raw[:-3].rstrip()
        return raw
    except Exception as e:
        return f"Error: {e}"

In [ ]:
def run_ui(template_name, num_rows, output_format, custom_instruction):
    if num_rows is None or num_rows < 1:
        num_rows = 10
    if num_rows > 500:
        num_rows = 500
    return generate_dataset(template_name or "Crop records", num_rows, output_format or "CSV", custom_instruction or "")

In [ ]:
# Gradio UI
with gr.Blocks(title="Agronomist Data Generator", theme=gr.themes.Soft()) as app:
    gr.Markdown("## Agronomist Synthetic Data Generator")
    with gr.Row():
        template = gr.Dropdown(
            choices=list(PROMPT_TEMPLATES.keys()),
            value="Crop records",
            label="Dataset type",
        )
        num_rows = gr.Number(value=10, minimum=1, maximum=500, label="Number of rows", precision=0)
        output_format = gr.Radio(choices=["CSV", "JSON"], value="CSV", label="Output format")
    custom_instruction = gr.Textbox(
        label="Custom instruction (only for 'Custom' dataset type)",
        placeholder="e.g. fertilizer applications with field_id, date, product, amount_kg",
        lines=2,
    )
    btn = gr.Button("Generate")
    output = gr.Textbox(label="Generated data", lines=15, max_lines=30)
    btn.click(
        fn=run_ui,
        inputs=[template, num_rows, output_format, custom_instruction],
        outputs=output,
    )
app.launch()